# ICVSP: pothole & speed-bump detector (merge → train → evaluate)

**No Google Drive access needed.** This notebook asks for no Google permissions:
- the project scripts are written into the Colab machine by the cells below,
- the public SBP-YOLO dataset is downloaded without signing in,
- results are downloaded to your computer as a zip at the end.

**To run:** Runtime → Change runtime type → **T4 GPU**, then **Runtime → Run all**.
Class IDs were verified on 29 Sep (0 = pothole, 1 = speed bump).

Everything on the Colab machine is deleted when the session ends, so **download the results zip (last cell) before closing**.

_Generated by `scripts/build_notebook.py`. Edit the files in `ICVSP/ai/` and regenerate; do not edit the embedded copies below._

## 1. Check the GPU and set up the project folder

In [ ]:
!nvidia-smi -L
%mkdir -p /content/ai/scripts /content/ai/configs /content/ai/runs
%cd /content/ai

## 2. Project scripts

In [ ]:
%%writefile /content/ai/requirements.txt
ultralytics>=8.3
requests
pyyaml
pillow
matplotlib


In [ ]:
%%writefile /content/ai/configs/classes.yaml
# Unified class scheme for the ICVSP road-hazard detector.
# Every source dataset is remapped onto these IDs by scripts/merge_datasets.py.
classes:
  0: pothole
  1: speed_bump

# Source class names are matched case-insensitively against these regexes.
# A source class that matches nothing is dropped (its boxes are removed, the image is kept).
mapping:
  pothole: 'pothole|pot[_ -]?hole|d40'
  speed_bump: 'speed[_ -]?(bump|breaker|hump)|^bump|hump|breaker|speedbump'

# Classes that must never be mapped even if a regex above matches them.
exclude: 'rumble|crack|d00|d10|d20|manhole'

# Class names for sources that ship without a data.yaml, keyed by folder name under data/raw/.
sources:
  # SBP-YOLO has no data.yaml. Roboflow exports number classes alphabetically, so 0 = pothole,
  # 1 = speed bump is the likely order. VERIFY with the "check class IDs" cell in the notebook.
  sbp_yolo:
    names: [pothole, speed_bump]


In [ ]:
%%writefile /content/ai/scripts/download_sbp.py
"""Download the public SBP-YOLO dataset without signing in to Google.

The authors share it as a public Google Drive folder of loose files (about 14,750 images and
labels), which is too many for `gdown --folder` (50-file limit). This script lists the folder
through Drive's public embedded view and downloads each file in parallel, with retries.
Files that already exist are skipped, so it is safe to re-run after an interruption.

Dataset: Liang et al., "SBP-YOLO", J. Real-Time Image Processing 23, 52 (2026).
         https://github.com/chuanqi1997/SBP-YOLO

Usage:  python scripts/download_sbp.py [--out /content/raw/sbp_yolo] [--workers 16]
"""
import argparse, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests

DATASET_FOLDER = '1anMQJlIeVCxH2xsk-cGJwVRb166ztyyJ'  # SBP-YOLO-Dataset/dataset (train/valid/test)
ENTRY = re.compile(r'<a href="https://drive.google.com/(drive/folders|file/d)/([^/"?]+)[^"]*"[^>]*>.*?'
                   r'flip-entry-title">([^<]+)', re.S)
session = requests.Session()


def list_folder(fid, prefix=''):
    html = session.get(f'https://drive.google.com/embeddedfolderview?id={fid}', timeout=60).text
    files = []
    for kind, item_id, name in ENTRY.findall(html):
        if kind == 'drive/folders':
            files += list_folder(item_id, f'{prefix}{name}/')
        else:
            files.append((f'{prefix}{name}', item_id))
    return files


def fetch(rel, fid, out: Path, tries=6):
    dst = out / rel
    if dst.exists() and dst.stat().st_size > 0:
        return 'skipped'
    dst.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(tries):
        try:
            r = session.get('https://drive.usercontent.google.com/download',
                            params={'id': fid, 'export': 'download'}, timeout=60)
            if r.status_code == 200 and not r.headers.get('content-type', '').startswith('text/html'):
                dst.write_bytes(r.content)
                return 'ok'
        except requests.RequestException:
            pass
        time.sleep(2 ** attempt)  # back off on rate limits (429) and transient errors
    return 'failed'


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--out', default='/content/raw/sbp_yolo', type=Path)
    ap.add_argument('--workers', type=int, default=16)
    a = ap.parse_args()

    print('Listing the public folder ...')
    files = list_folder(DATASET_FOLDER)
    print(f'{len(files)} files found; downloading with {a.workers} workers')
    stats, failed = {'ok': 0, 'skipped': 0, 'failed': 0}, []
    with ThreadPoolExecutor(a.workers) as pool:
        jobs = {pool.submit(fetch, rel, fid, a.out): rel for rel, fid in files}
        for n, job in enumerate(as_completed(jobs), 1):
            res = job.result()
            stats[res] += 1
            if res == 'failed':
                failed.append(jobs[job])
            if n % 1000 == 0 or n == len(files):
                print(f'  {n}/{len(files)}  {stats}')
    if failed:
        (a.out / 'failed.txt').write_text('\n'.join(failed))
        print(f'{len(failed)} files failed (listed in failed.txt). Re-run this script to retry them.')
    else:
        print('All files downloaded.')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/ai/scripts/merge_datasets.py
"""Merge YOLO-format road-hazard datasets into one dataset with the unified ICVSP classes.

For every source folder under data/raw/<source>/ that contains a data.yaml (or a classes.txt),
the script:
  1. remaps source class IDs to the unified scheme in configs/classes.yaml,
  2. drops boxes whose class does not map (e.g. rumble strips, cracks),
  3. removes near-duplicate images across all sources (perceptual dHash),
  4. writes data/merged/{train,val,test}/{images,labels} and data/merged/data.yaml.

Splits: a source's own test split is kept as test, so results stay comparable with the
source paper; everything else is re-split train/val 85/15 per source.

Usage:  python scripts/merge_datasets.py [--raw data/raw] [--out data/merged]
"""
import argparse, hashlib, json, random, re, shutil
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from PIL import Image

ROOT = Path(__file__).resolve().parents[1]
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def load_scheme():
    cfg = yaml.safe_load((ROOT / 'configs/classes.yaml').read_text())
    names = [cfg['classes'][i] for i in sorted(cfg['classes'])]
    rules = [(names.index(k), re.compile(v, re.I)) for k, v in cfg['mapping'].items()]
    return names, rules, re.compile(cfg['exclude'], re.I)


def source_names(src: Path):
    override = (yaml.safe_load((ROOT / 'configs/classes.yaml').read_text()).get('sources') or {}).get(src.name)
    if override:
        return list(override['names'])
    for y in list(src.rglob('data.yaml')) + list(src.rglob('*.yaml')):
        try:
            d = yaml.safe_load(y.read_text())
        except Exception:
            continue
        if isinstance(d, dict) and 'names' in d:
            n = d['names']
            return [n[k] for k in sorted(n)] if isinstance(n, dict) else list(n)
    for t in src.rglob('classes.txt'):
        return [l.strip() for l in t.read_text().splitlines() if l.strip()]
    raise SystemExit(f'No data.yaml / classes.txt with class names found in {src}')


def build_map(names, rules, exclude):
    m = {}
    for i, n in enumerate(names):
        if exclude.search(n):
            continue
        for uid, rx in rules:
            if rx.search(n):
                m[i] = uid
                break
    return m


def dhash(path, size=8):
    with Image.open(path) as im:
        im = im.convert('L').resize((size + 1, size))
        px = im.tobytes()  # one byte per pixel in 'L' mode
    bits = ''.join('1' if px[r * (size + 1) + c] > px[r * (size + 1) + c + 1] else '0'
                   for r in range(size) for c in range(size))
    return int(bits, 2)


def split_of(img: Path):
    parts = {p.lower() for p in img.parts}
    if 'test' in parts:
        return 'test'
    if 'valid' in parts or 'val' in parts:
        return 'val'
    return 'train'


def label_for(img: Path):
    # YOLO convention: .../images/x.jpg <-> .../labels/x.txt ; fall back to same folder
    parts = list(img.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == 'images':
            parts[i] = 'labels'
            cand = Path(*parts).with_suffix('.txt')
            if cand.exists():
                return cand
    cand = img.with_suffix('.txt')
    return cand if cand.exists() else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--raw', default=ROOT / 'data/raw', type=Path)
    ap.add_argument('--out', default=ROOT / 'data/merged', type=Path)
    ap.add_argument('--seed', default=0, type=int)
    args = ap.parse_args()
    random.seed(args.seed)
    names, rules, exclude = load_scheme()

    if args.out.exists():
        shutil.rmtree(args.out)
    for s in ('train', 'val', 'test'):
        (args.out / s / 'images').mkdir(parents=True)
        (args.out / s / 'labels').mkdir(parents=True)

    seen = {}
    report = {'sources': {}, 'duplicates_removed': 0}
    counts = defaultdict(Counter)

    for src in sorted(p for p in args.raw.iterdir() if p.is_dir()):
        src_names = source_names(src)
        cmap = build_map(src_names, rules, exclude)
        imgs = sorted(p for p in src.rglob('*') if p.suffix.lower() in IMG_EXT)
        rep = {'source_classes': src_names, 'mapping': {src_names[k]: names[v] for k, v in cmap.items()},
               'images': 0, 'kept': 0, 'dropped_boxes': 0}
        pending = []
        for img in imgs:
            rep['images'] += 1
            try:
                h = dhash(img)
            except Exception:
                continue
            if h in seen:
                report['duplicates_removed'] += 1
                continue
            seen[h] = img
            lab = label_for(img)
            lines = []
            if lab:
                for ln in lab.read_text().splitlines():
                    f = ln.split()
                    if len(f) < 5:
                        continue
                    cid = int(float(f[0]))
                    if cid in cmap:
                        lines.append(' '.join([str(cmap[cid])] + f[1:5]))
                    else:
                        rep['dropped_boxes'] += 1
            pending.append((img, lines, split_of(img)))

        # re-split non-test images 85/15 per source
        rest = [x for x in pending if x[2] != 'test']
        random.shuffle(rest)
        n_val = round(0.15 * len(rest))
        final = [(i, l, 'val') for i, l, _ in rest[:n_val]] + [(i, l, 'train') for i, l, _ in rest[n_val:]] \
            + [x for x in pending if x[2] == 'test']

        tag = re.sub(r'[^a-z0-9]+', '_', src.name.lower())
        for img, lines, split in final:
            stem = f'{tag}_{hashlib.md5(str(img).encode()).hexdigest()[:10]}'
            shutil.copy2(img, args.out / split / 'images' / f'{stem}{img.suffix.lower()}')
            (args.out / split / 'labels' / f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
            for l in lines:
                counts[split][names[int(l.split()[0])]] += 1
            counts[split]['images'] += 1
            rep['kept'] += 1
        report['sources'][src.name] = rep

    (args.out / 'data.yaml').write_text(yaml.safe_dump({
        'path': str(args.out), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
        'names': {i: n for i, n in enumerate(names)}}, sort_keys=False))
    report['counts'] = {k: dict(v) for k, v in counts.items()}
    (args.out / 'merge_report.json').write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/ai/scripts/train.py
"""Train the ICVSP pothole / speed-bump detector on the merged dataset.

Defaults are sized for a 16 GB Apple-silicon laptop (MPS). On Colab, pass --device 0 --batch 32.

Usage:  python scripts/train.py [--model yolo11n.pt] [--epochs 50] [--fraction 1.0]
        --fraction < 1.0 trains on a random subset (used for the learning curve).
"""
import argparse
from pathlib import Path

import torch
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parents[1]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', default=ROOT / 'data/merged/data.yaml')
    ap.add_argument('--model', default='yolo11n.pt')
    ap.add_argument('--epochs', type=int, default=50)
    ap.add_argument('--imgsz', type=int, default=640)
    ap.add_argument('--batch', type=int, default=16)
    ap.add_argument('--fraction', type=float, default=1.0)
    ap.add_argument('--device', default='mps' if torch.backends.mps.is_available() else 'cpu')
    ap.add_argument('--patience', type=int, default=15, help='stop early after this many epochs without improvement')
    ap.add_argument('--name', default=None)
    a = ap.parse_args()

    name = a.name or f'{Path(a.model).stem}_e{a.epochs}_f{int(a.fraction * 100)}'
    YOLO(a.model).train(
        data=str(a.data), epochs=a.epochs, imgsz=a.imgsz, batch=a.batch, device=a.device,
        fraction=a.fraction, project=str(ROOT / 'runs'), name=name, exist_ok=True,
        patience=a.patience, workers=4, seed=0, plots=True,
    )


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/ai/scripts/evaluate.py
"""Evaluate a trained model and apply the "is the merge enough?" decision rule.

Evaluates on the public test split and, when it exists, on the Egyptian test set
(data/egypt_test/data.yaml). The decision is taken on the EGYPTIAN set only; the public
number is a sanity check.

Decision rule (agreed before testing, see proposal §AI plan):
  speed_bump recall >= 0.75 and precision >= 0.70  -> merge is enough
  0.50 <= recall < 0.75                            -> targeted top-up of the failure cases
  recall < 0.50                                    -> build a proper Egyptian training set

Usage:  python scripts/evaluate.py --weights runs/<run>/weights/best.pt
"""
import argparse, json
from pathlib import Path

import torch
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parents[1]
ENOUGH_RECALL, ENOUGH_PRECISION, TOPUP_RECALL = 0.75, 0.70, 0.50


def per_class(metrics, names):
    out = {}
    present = list(metrics.box.ap_class_index)  # class_result() indexes by position in this list
    for i, n in names.items():
        p, r, map50, map5095 = metrics.box.class_result(present.index(i)) if i in present else (0, 0, 0, 0)
        out[n] = {'precision': round(float(p), 3), 'recall': round(float(r), 3),
                  'mAP50': round(float(map50), 3), 'mAP50-95': round(float(map5095), 3)}
    return out


def verdict(cls):
    sb = cls.get('speed_bump', {})
    r, p = sb.get('recall', 0), sb.get('precision', 0)
    if r >= ENOUGH_RECALL and p >= ENOUGH_PRECISION:
        return 'ENOUGH: public merge generalises to Egyptian speed bumps'
    if r >= TOPUP_RECALL:
        return 'TOP-UP: collect 50-100 images of the failure cases only'
    return 'NOT ENOUGH: build a proper Egyptian training set'


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--weights', required=True)
    ap.add_argument('--device', default='mps' if torch.backends.mps.is_available() else 'cpu')
    ap.add_argument('--public-data', default=ROOT / 'data/merged/data.yaml', type=Path)
    ap.add_argument('--egypt-data', default=ROOT / 'data/egypt_test/data.yaml', type=Path)
    ap.add_argument('--imgsz', type=int, default=640, help='use the same size the model was trained at')
    a = ap.parse_args()
    model = YOLO(a.weights)
    results = {}
    sets = {'public_test': a.public_data, 'egypt_test': a.egypt_data}
    for tag, yml in sets.items():
        if not yml.exists():
            results[tag] = 'missing'
            continue
        run = Path(a.weights).parents[1].name
        m = model.val(data=str(yml), split='test', device=a.device, plots=True, imgsz=a.imgsz,
                      project=str(ROOT / 'runs'), name=f'eval_{tag}_{run}', exist_ok=True)
        results[tag] = {'overall_mAP50': round(float(m.box.map50), 3), 'per_class': per_class(m, model.names)}
    if isinstance(results.get('egypt_test'), dict):
        results['decision'] = verdict(results['egypt_test']['per_class'])
    else:
        results['decision'] = 'PENDING: no Egyptian test set yet (see README, step 3)'
    out = Path(a.weights).parents[1] / 'evaluation.json'
    out.write_text(json.dumps(results, indent=2))
    print(json.dumps(results, indent=2))


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/ai/scripts/predict_clips.py
"""Quick visual check: run the detector on the team's Egyptian phone clips.

Saves each clip with the model's boxes drawn on it, plus a per-clip summary of when it saw a
speed bump or pothole. This shows WHAT the model detects, but it is not a measurement:
it cannot tell you what the model missed. For recall/precision numbers, label the frames
and use evaluate.py (README, step 3).

Usage:  python scripts/predict_clips.py --weights runs/v1_e150/weights/best.pt [--conf 0.25]
"""
import argparse, csv
from pathlib import Path

import cv2
import torch
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parents[1]
VIDEO = {'.mov', '.mp4', '.m4v', '.avi'}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--weights', required=True)
    ap.add_argument('--clips', default=ROOT / 'data/egypt_test/clips', type=Path)
    ap.add_argument('--out', default=ROOT / 'runs/egypt_clips', type=Path)
    ap.add_argument('--conf', type=float, default=0.25)
    ap.add_argument('--imgsz', type=int, default=640)
    ap.add_argument('--device', default='0' if torch.cuda.is_available() else
                    'mps' if torch.backends.mps.is_available() else 'cpu')
    a = ap.parse_args()

    model = YOLO(a.weights)
    rows = []
    for clip in sorted(p for p in a.clips.iterdir() if p.suffix.lower() in VIDEO):
        hits = {n: [] for n in model.names.values()}
        cap = cv2.VideoCapture(str(clip)); fps = cap.get(cv2.CAP_PROP_FPS) or 30.0; cap.release()
        frames = 0
        for r in model.predict(source=str(clip), conf=a.conf, imgsz=a.imgsz, device=a.device, stream=True,
                               save=True, project=str(a.out), name=clip.stem, exist_ok=True, verbose=False):
            t = frames / fps
            for c, s in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()):
                hits[model.names[int(c)]].append((t, s))
            frames += 1
        for cls, h in hits.items():
            rows.append({'clip': clip.name, 'class': cls, 'frames_with_detection': len({round(t, 3) for t, _ in h}),
                         'first_seen_s': round(min(t for t, _ in h), 1) if h else '',
                         'last_seen_s': round(max(t for t, _ in h), 1) if h else '',
                         'max_confidence': round(max(s for _, s in h), 2) if h else ''})
        seen = ', '.join(f'{c}: {len(h)} boxes' for c, h in hits.items())
        print(f'{clip.name}: {frames} frames ({frames / fps:.0f} s) -> {seen}')
    a.out.mkdir(parents=True, exist_ok=True)
    with open(a.out / 'summary.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]) if rows else ['clip'])
        w.writeheader(); w.writerows(rows)
    print(f'\nAnnotated videos and summary.csv saved in {a.out}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/ai/scripts/extract_frames.py
"""Turn the team's short phone clips into frames for labelling.

Put clips in data/egypt_test/clips/<area>_<person>_<n>.mov (the area prefix is used to
split by location, never by frame). Frames go to data/egypt_test/frames/<clip-name>/.

Usage:  python scripts/extract_frames.py [--fps 2]
"""
import argparse, subprocess
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--fps', type=float, default=2)
    a = ap.parse_args()
    clips = sorted(p for p in (ROOT / 'data/egypt_test/clips').iterdir()
                   if p.suffix.lower() in {'.mov', '.mp4', '.m4v'})
    for clip in clips:
        out = ROOT / 'data/egypt_test/frames' / clip.stem
        out.mkdir(parents=True, exist_ok=True)
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', str(clip), '-vf', f'fps={a.fps}',
                        '-q:v', '2', str(out / f'{clip.stem}_%04d.jpg')], check=True)
        print(f'{clip.name}: {len(list(out.glob("*.jpg")))} frames')


if __name__ == '__main__':
    main()


## 2b. Install packages (every new session; takes about a minute)

In [ ]:
!pip -q install -r /content/ai/requirements.txt
import ultralytics; print("ultralytics", ultralytics.__version__, "installed")

## 3. Download SBP-YOLO (7,375 images + labels, public, no sign-in)
Takes roughly 10–20 minutes. If it reports failed files (usually Drive rate limits), just run the cell again; it only fetches what is missing.

In [ ]:
!python scripts/download_sbp.py --out /content/raw/sbp_yolo
!du -sh /content/raw/sbp_yolo

## 4. ⚠️ Check the class IDs (do this once)
SBP-YOLO ships without class names. `configs/classes.yaml` assumes **class 0 = pothole, class 1 = speed bump**.
Look at the red boxes below. If the rows are the other way round, swap the two names under `sources: sbp_yolo` in the classes.yaml cell above, re-run that cell, then continue.

In [ ]:
import random, pathlib
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
root = pathlib.Path('/content/raw/sbp_yolo/train')
by_class = {0: [], 1: []}
for lab in root.glob('labels/*.txt'):
    for ln in lab.read_text().splitlines():
        c = int(ln.split()[0])
        if c in by_class and len(by_class[c]) < 50:
            by_class[c].append(lab)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, c in enumerate(by_class):
    for ax, lab in zip(axes[row], random.sample(by_class[c], 3)):
        img = Image.open(root / 'images' / (lab.stem + '.jpg')); w, h = img.size
        ax.imshow(img); ax.axis('off'); ax.set_title(f'class {c}')
        for ln in lab.read_text().splitlines():
            k, x, y, bw, bh = map(float, ln.split()[:5])
            if int(k) == c:
                ax.add_patch(patches.Rectangle(((x - bw / 2) * w, (y - bh / 2) * h), bw * w, bh * h,
                                               fill=False, color='red', lw=2))
plt.tight_layout(); plt.show()

## 5. Merge into the unified classes (`pothole`, `speed_bump`)
Check the report: every source class must map to the right unified class.

In [ ]:
!python scripts/merge_datasets.py --raw /content/raw --out /content/merged

## 6. Experiments
Run **one experiment per Colab session** (free sessions time out). Each new session: run steps 1–5 again (about 20 min), then one experiment below, then step 9 to download.

| Run | What changes | Time on a T4 | Status |
|---|---|---|---|
| v0_public_merge | baseline: 50 epochs, 640 px | ~1 h | ✅ done 29 Sep |
| v1_e150 | train longer: 150 epochs | ~3 h | |
| v2_e150_960 | + bigger images (960 px) for small potholes | ~6 h, may need Colab Pro or 2 sessions | |

### 6a. v0 baseline (done, kept for reference; uncomment only to reproduce)

In [ ]:
# !python scripts/train.py --data /content/merged/data.yaml --device 0 --batch 32 --epochs 50 --name v0_public_merge
# !python scripts/evaluate.py --weights runs/v0_public_merge/weights/best.pt --device 0 --public-data /content/merged/data.yaml

### 6b. v1: 150 epochs
The baseline was still improving at epoch 50. `patience 30` stops early if nothing improves for 30 epochs.

In [ ]:
import pathlib
assert pathlib.Path('/content/merged/data.yaml').exists(), 'Run steps 3-5 (download + merge) in this session first.'
!python scripts/train.py --data /content/merged/data.yaml --device 0 --batch 32 --epochs 150 --patience 30 --name v1_e150
!python scripts/evaluate.py --weights runs/v1_e150/weights/best.pt --device 0 --public-data /content/merged/data.yaml

### 6c. v2: 150 epochs at 960 px
Helps small, distant potholes, but runs about 2× slower on the device too; the Hailo speed test decides if it is usable.

In [ ]:
# Run in its own session: uncomment both lines, and comment out the v1 cell above.
# !python scripts/train.py --data /content/merged/data.yaml --device 0 --batch 16 --epochs 150 --patience 30 --imgsz 960 --name v2_e150_960
# !python scripts/evaluate.py --weights runs/v2_e150_960/weights/best.pt --device 0 --imgsz 960 --public-data /content/merged/data.yaml

## 7. (Done) Learning curve
Already run on 29 Sep (25/50/75 %). Uncomment only to reproduce.

In [ ]:
# for f in (0.25, 0.5, 0.75):
#     !python scripts/train.py --data /content/merged/data.yaml --device 0 --batch 32 --epochs 50 --fraction {f}

## 8. Download the results
Trained models, metrics, confusion matrices and merge report (not the dataset, not the clip videos). Unzip into `ICVSP/ai/` and run `python scripts/compare_runs.py` to add the run to `results/experiments.md`.

In [ ]:
!cp /content/merged/merge_report.json runs/ 2>/dev/null; cd /content/ai && zip -qr /content/icvsp_results.zip runs -x 'runs/egypt_clips/*'
from google.colab import files
files.download('/content/icvsp_results.zip')

## 9. Quick check on Egyptian clips
Shows what the model detects on your own phone clips: boxes drawn on each video, plus `summary.csv` with when it saw a bump.

⚠️ This is a **visual sanity check, not a measurement**: it shows detections but not what the model missed. The real accuracy numbers come from labelling frames (README, step 3).

This step only needs steps 1–2, not the dataset download. Upload the clips (1080p, short) when asked. It uses the model trained in this session if there is one; otherwise upload a `best.pt` from `ICVSP/ai/runs/<run>/weights/`.

In [ ]:
import pathlib, shutil
from google.colab import files
clips = pathlib.Path('/content/ai/data/egypt_test/clips'); clips.mkdir(parents=True, exist_ok=True)
print('Select your clips:')
for name in files.upload():
    shutil.move(name, clips / name)
trained = sorted(pathlib.Path('/content/ai/runs').glob('v*/weights/best.pt'))
if trained:
    WEIGHTS = str(trained[-1])
else:
    print('No model trained in this session. Select a best.pt:')
    WEIGHTS = '/content/ai/' + next(iter(files.upload()))
print('Using', WEIGHTS)
!python scripts/predict_clips.py --weights "$WEIGHTS" --conf 0.25
!cd /content/ai/runs && zip -qr /content/egypt_clips_check.zip egypt_clips
files.download('/content/egypt_clips_check.zip')